In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.interpolate import interp1d
from scipy.ndimage import label
import optuna
from collections import deque
from loader import * 
from scripts.processing import *
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.model_selection import LeaveOneOut

# Objective Function

In [ ]:
def multi_objective(trial, measured_vs, average_fxs, average_fys, des_predicted_v, durations):

    delay = trial.suggest_float('delay', 0.01, 0.07) #davor: 0.07
    

    K_fy = trial.suggest_float('K_fy', -15.141400596291194e-07, -5.141400596291194e-07)#K_fy = trial.suggest_float('K_fy', -0.004, 1)
    K_fx = trial.suggest_float('K_fx', -0.002, 0.2)
    Kp = trial.suggest_float('Kp', 0.001, 20)
    Kd = trial.suggest_float('Kd', 0, 5e-16)#trial.suggest_float('Kd', 0, 0.2)

    n_trials = len(measured_vs)
    all_errors = []

    for n in range(n_trials):

        measured_v = measured_vs[n]
        des_speed = des_predicted_v[n]
        duration = durations[n]

        delta_t = duration / (len(measured_v))
        delay_samples = round(delay / delta_t)

        fy = [0 if i < 0 else i for i in average_fys[n]]
        fx = [0 if y < 0 else x
              for x, y in zip(average_fxs[n], average_fys[n])]
       # fy = [y for y in average_fys[n]]
      #  fx = [y for y in average_fxs[n]]
        
        measured_v = [des_speed if y < 10 else v
             for v, y in zip(measured_vs[n], average_fys[n])]

        start_index = int(90 * len(measured_v) / 100)

        predicted_v = np.zeros(len(measured_v))

        speed = des_speed
        predicted_v[start_index] = speed

        fx_all = deque(maxlen=delay_samples)
        fy_all = deque(maxlen=delay_samples)

        fx_all.append(0)
        fy_all.append(0)

        e_all = deque(maxlen=3)
        e = 0
        e_all.append(e)
       
        

        for k in range(len(measured_v) - 1):

            i = (start_index + k) % len(measured_v)
            next_i = (i + 1) % len(measured_v)

            

            if fy[next_i] <10: #this must match average v above
                speed_next = des_speed

                dfxdt = (fx[i] - fx[i - 1]) / delta_t
                dfydt = (fy[i] - fy[i - 1]) / delta_t

                fx_all.append(dfxdt)
                fy_all.append(dfydt)
                e = des_speed - speed
                e_all.append(e)
       

            else:
                dfxdt = (fx[i] - fx[i - 1]) / delta_t
                dfydt = (fy[i] - fy[i - 1]) / delta_t

                fx_all.append(dfxdt)
                fy_all.append(dfydt)

                e = des_speed - speed
                e_all.append(e)

             
                dedt = (e_all[1] - e_all[0]) / delta_t

                grf_part = K_fx * fx_all[0] + K_fy * fy_all[0]
                pd_part = Kp * e_all[1] + Kd * dedt

                speed_next = speed + grf_part + pd_part


            predicted_v[next_i] = speed_next
            speed = speed_next

        MSE = np.square(np.subtract(measured_v, predicted_v)).mean()
        all_errors.append(MSE)

    return np.mean(all_errors)

In [ ]:
def final(measured_v, average_fx, average_fy, params, des_speed, duration):

    delay = params['delay']
    K_fy = params['K_fy']
    Kp = params['Kp']
    Kd = params['Kd']
    K_fx = params['K_fx']

    delta_t =  duration / (len(measured_v))
   # print(delay / delta_t)
    print(duration, delta_t)
    delay_samples = round(delay / delta_t)
    #print(delay_samples, delta_t)

    start_index = int(90 * len(measured_v) / 100)

    fy = [0 if y < 0 else y
      for y in average_fy]
   # fy = [y for y in average_fy]
  #  fx = [y for y in average_fx]

    fx = [0 if y < 0 else x
      for x, y in zip(average_fx, average_fy)]

    measured_v = [des_speed if y < 10 else v
             for v, y in zip(measured_v, average_fy)]

    speed = des_speed
    predicted_v = np.zeros(len(measured_v))
    predicted_v[start_index] = speed

    fx_all = deque(maxlen=delay_samples)
    fy_all = deque(maxlen=delay_samples)

    fx_all.append(0)
    fy_all.append(0)

    e_all = deque(maxlen=3)
    e = 0
    e_all.append(e)
    

    for k in range(len(measured_v) - 1):

        i = (start_index + k) % len(measured_v)
        next_i = (i + 1) % len(measured_v)

        
        if fy[next_i] <10: #this must match average v above
                speed_next = des_speed

                dfxdt = (fx[i] - fx[i - 1]) / delta_t
                dfydt = (fy[i] - fy[i - 1]) / delta_t

                fx_all.append(dfxdt)
                fy_all.append(dfydt)
                e = des_speed - speed
                e_all.append(e)
       

        else:
                dfxdt = (fx[i] - fx[i - 1]) / delta_t
                dfydt = (fy[i] - fy[i - 1]) / delta_t

                fx_all.append(dfxdt)
                fy_all.append(dfydt)

                e = des_speed - speed
                e_all.append(e)

             
                dedt = (e_all[1] - e_all[0]) / delta_t

                grf_part = K_fx * fx_all[0] + K_fy * fy_all[0]
                pd_part = Kp * e_all[1] + Kd * dedt

                speed_next = speed + grf_part + pd_part

        predicted_v[next_i] = speed_next
        speed = speed_next

    return predicted_v

# Load data

In [ ]:
exp_data_folder = Path("./data")
dataset = DatasetGrail(exp_data_folder, "./config.xlsx")
dataset = dataset.get_subset(
    task=["walking"], 
)
index = dataset.create_index()

## Initialization

In [ ]:
x0 = {'delay': 0.05052497232666044,
 'K_fy': -9.141400596291194e-07,
 'K_fx': 4.3300646115648653e-06,
 'Kp': 0.6309665948387769,
 'Kd': 2.1007497833641600261e-17}

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
speeds = np.unique(dataset.index.speed)
speeds

## Leave one speed out cross-validation

optimize on 4 predicted_v (all participants), test on the 5th one (all participants)

In [ ]:
loo = LeaveOneOut()
loo.get_n_splits(speeds)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import optuna

for seed in [0, 1, 100]:

    out_dir = Path("LOSO_seed{}".format(seed))
    out_dir.mkdir(exist_ok=True)

    test_predicted_v = []
    test_errors = []
    train_errors = []
    all_params = []
    all_test_errors = []
    for i, (train_index, test_index) in enumerate(loo.split(speeds)):
    
        print("fold", i)

        trainset = dataset.get_subset(speed=list(speeds[train_index]))
        testset = dataset.get_subset(speed=list(speeds[test_index]))

        vs, fys, fxs, durs, des_vs = [], [], [], [], []

        for p in trainset:
            cycles, dur = get_data(p)

            vs.append(cycles.speedL)
            fys.append(cycles.FyL)
            fxs.append(cycles.FxL)
            durs.append(dur)
            des_vs.append(p.index.speed[0])

        func_multi = lambda trial: multi_objective(
            trial, vs, fxs, fys, des_vs, durs
        )

        sampler = optuna.samplers.CmaEsSampler(x0=x0, seed=seed)
        study_multi = optuna.create_study(sampler=sampler)
        study_multi.optimize(func_multi, n_trials=4000)

        train_errors.append(study_multi.best_value)
        all_params.append(study_multi.best_params)
        print(study_multi.best_params)

        speed_errors = []
        speed_predicted_v = []

        for j, p in enumerate(testset):

            participant_id = p.index.id[0]
            trial_speed = p.index.speed[0]

            cycles, dur = get_data(p)

            test_speed = final(
                cycles.speedL,
                cycles.FxL,
                cycles.FyL,
                study_multi.best_params,
                trial_speed,
                dur
            )

            measured_speed = np.asarray(cycles.speedL)
            predicted_speed = np.asarray(test_speed)
            fy = np.asarray(cycles.FyL)

            stance_mask = fy >= 10

            error = np.sqrt(
                np.mean((measured_speed[stance_mask] - predicted_speed[stance_mask]) ** 2)
            )
    
            speed_errors.append(error)
            speed_predicted_v.append(test_speed)

            df_pred = pd.DataFrame({
                "time_index": np.arange(len(test_speed)),
                "measured_speed": measured_speed,
                "predicted_speed": predicted_speed,
                "FyL": fy,
                "stance": stance_mask,
                "desired_speed": trial_speed,
                "stance_rmse": error
            })

            df_pred.to_csv(
                out_dir / f"{participant_id}_speed{trial_speed}_predicted_speed.csv",
                index=False
            )

    # save participant-level result
        test_errors.append(np.mean(speed_errors))
        all_test_errors.extend(speed_errors)
        test_predicted_v.append(speed_predicted_v)

        print(
            f"left-out participant: {trial_speed}, "
            f"mean stance RMSE: {np.mean(speed_errors):.4f}"
        )
    print(np.mean(all_test_errors))

np.mean(all_test_errors)

In [ ]:
np.std(all_test_errors)

In [ ]:
errors = pd.DataFrame([train_errors, [t**2 for t in test_errors], test_errors]).T
errors.columns = ['trainMSE', 'testMSE', 'testRMSE']
#errors.to_csv('results/errors_train1.8_test1.8.csv')

In [ ]:
errors

In [ ]:
errors.mean()

## visualize the results

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

folder1 = Path("./LOSO_seed0/")

for file1 in folder1.glob("*.csv"):
    

    df1 = pd.read_csv(file1)


    plt.figure(figsize=(8, 4))
    plt.plot(df1.measured_speed, label="measured", color="black")
    plt.plot(df1.predicted_speed, label="seed1")

    plt.title(file1.name)
    plt.legend()
    plt.show()

In [ ]:
plt.figure(figsize=(7, 4))

# --- First subplot ---
plt.subplot(1, 2, 1)
plt.plot(cycles.speedL, label='measured', color="#00316A")
plt.plot(tests['1.6_2'], label='predicted, unseen speed', color="#81C063")
plt.plot(pd.read_csv('S06_16_lopo.csv', header=None).values[0], label='predicted, unseen participant',
        color="#E69F00")
#plt.title('participant 4', pad=-200)
plt.ylim(1.548, 1.625)
plt.xlabel("% of stride", fontsize=12)
plt.text(0.5, 0.95, 'participant 4', transform=plt.gca().transAxes,
         ha='center', va='bottom', fontsize=12)
plt.ylabel("belt speed", fontsize=12)
plt.hlines(1.6, 0, 100, colors='gray', linestyles='dashed', label='desired speed')
sns.despine()

# --- Second subplot ---
plt.subplot(1, 2, 2)
plt.plot(cycles2.speedL, label='measured', color=p[0])
plt.plot(tests['1.6_7'], label='predicted, unseen speed', color=p[-1])
plt.plot(pd.read_csv('S10_16_lopo.csv', header=None).values[0], label='predicted, unseen participant',
        color=p[2])
#plt.title('participant 8', pad=-200)
plt.hlines(1.6, 0, 100, colors='gray', linestyles='dashed', label='desired speed')
plt.ylim(1.548, 1.625)
plt.text(0.5, 0.95, 'participant 8', transform=plt.gca().transAxes,
         ha='center', va='bottom', fontsize=12)
plt.xlabel("% of stride", fontsize=12)
#plt.ylabel("belt speed", fontsize=12)

sns.despine()

# --- Combine legends ---
fig = plt.gcf()
handles, labels = plt.gca().get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.05), fontsize=11  )

# --- Adjust layout ---
#plt.tight_layout()
plt.subplots_adjust(bottom=0.22)


#plt.show()
plt.savefig('abstract_fig.svg', bbox_inches='tight')